# 06 — Demo: modeli, predikcija i rezultati

Ova sveska približava projekat bez pokretanja celog pipeline-a.

1. prikaz zbirnog poredjenja scratch vs transfer
2. confusion matrix / bar chart (iz `results/plots/`)
3. ucitavanje jednog checkpointa i predikcija na nekoliko slika

**Tim:** Irina Marko, Nikola Lazarević

In [ ]:
from pathlib import Path
import json
import importlib.util

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image

ROOT = Path('.').resolve()
if not (ROOT / 'results' / 'comparison.txt').exists():
    ROOT = ROOT.parent

print('ROOT:', ROOT)
print((ROOT / 'results' / 'comparison.txt').read_text(encoding='utf-8'))

## Graficko poredjenje modela

Ako nema `comparison_bars.png`, pokrenuti: `python scripts/06_eval_plots.py`

In [ ]:
from IPython.display import Image as IPyImage, display

bars = ROOT / 'results' / 'plots' / 'comparison_bars.png'
if bars.exists():
    display(IPyImage(filename=str(bars)))
else:
    print('nema comparison_bars.png — pokreni scripts/06_eval_plots.py')

## Confusion matrix (primer: category / transfer)

Ostale matrice su u `results/plots/*_confusion.png`.

In [ ]:
cm_path = ROOT / 'results' / 'plots' / 'category_transfer_confusion.png'
if cm_path.exists():
    display(IPyImage(filename=str(cm_path)))
else:
    print('nema matrice — pokreni scripts/06_eval_plots.py')

## Ucitavanje modela i predikcija

Koristimo transfer model za kategoriju (najbolji rezultat na tom zadatku).

In [ ]:
spec = importlib.util.spec_from_file_location(
    'train05', ROOT / 'scripts' / '05_train_models.py'
)
train05 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train05)

device = torch.device('cpu')
ckpt = torch.load(
    ROOT / 'models' / 'category_transfer_best.pt',
    map_location=device,
    weights_only=False,
)
class_names = ckpt['class_names']
model = train05.build_transfer_model(len(class_names))
model.load_state_dict(ckpt['state_dict'])
model.eval()
print('klase:', class_names)

In [ ]:
test_df = pd.read_csv(ROOT / 'dataset' / 'splits' / 'category_test.csv')
sample = test_df.sample(6, random_state=7)
tfm = train05.make_transforms(False)

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    path = ROOT / row['image_path']
    img = Image.open(path).convert('RGB')
    x = tfm(img).unsqueeze(0)
    with torch.no_grad():
        pred_idx = int(model(x).argmax(dim=1).item())
    pred = class_names[pred_idx]
    true = row['label']
    ok = 'OK' if pred == true else 'X'
    ax.imshow(img)
    ax.set_title(f'{ok}\ntrue={true}\npred={pred}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Kratki zakljucci

- transfer bolji na category i subcategory
- scratch bolji na color (ImageNet features manje osetljive na boju)
- subcategory je najtezi zadatak (30 klasa, slicne siluete)

Vise detalja u `DOKUMENTACIJA.md`.